<a href="https://colab.research.google.com/github/emgakii001/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emgakii001/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
!git clone https://github.com/emgakii001/flyrank-ml-internship.git


Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 119 (delta 34), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 1.82 MiB | 16.08 MiB/s, done.
Resolving deltas: 100% (34/34), done.


In [18]:
print("Files here:", os.listdir("."))

Files here: ['notebooks', 'data', 'requirements.txt', 'README.md', 'skills', 'flyrank-ml-internship', '.gitignore', 'scripts', 'CLAUDE.md', 'DATA_USE.md', '.git', 'submission', 'work', 'AGENTS.md', 'outputs', '.github', 'LICENSE', 'docs', 'SETUP.md', 'GUIDE.md']


In [19]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship/flyrank-ml-internship


In [20]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [21]:
print(df["trend_direction"].value_counts())

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 1. My lane (or freestyle) and why

Lane: Refresh / Content Opportunity Scoring

I'm choosing this lane because it produces a clear, prioritized action list, which content pages a reviewer should look at first rather than just describing patterns. This mirrors work I understand from bookkeeping: reviewing a portfolio of accounts and deciding which ones need attention first, given limited reviewer time.

In [22]:
# Section 1 — record the lane choice
LANE = "Lane 2: Refresh / Content Opportunity Scoring"
print("Selected lane:", LANE)

# Quick preview supporting the choice (full numbers come in Section 3)
preview_declining = (df["trend_direction"] == "down").sum()
print(f"Preview: {preview_declining} pages currently trending down — enough volume to justify this lane")# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Selected lane: Lane 2: Refresh / Content Opportunity Scoring
Preview: 16262 pages currently trending down — enough volume to justify this lane


## 2. The question: decision, action, cost of a wrong call


Decision: Which content pages should be reviewed first, out of thousands, given a reviewer has limited hours per week?
Who acts: A content strategist or SEO reviewer at a client, deciding what to refresh/expand/prune this week.
Action taken: Reviewer opens the top-ranked pages in the queue and either refreshes, expands, protects, prunes, or monitors each one.
Cost of a wrong call: Two kinds: (1) reviewer spends time on a page that didn't need attention (wasted hours), (2) a genuinely declining page gets missed and keeps losing visibility (lost traffic/revenue for the client).

In [23]:
# Section 2 — quantify the scale of the decision problem
total_pages = len(df)
declining_pages = (df["trend_direction"] == "down").sum()

# A realistic weekly review capacity (adjust this assumption as you like)
reviewer_capacity_per_week = 50

weeks_to_review_all_declining = declining_pages / reviewer_capacity_per_week

print(f"Total pages: {total_pages}")
print(f"Declining pages: {declining_pages}")
print(f"If a reviewer checks {reviewer_capacity_per_week} pages/week, "
      f"reviewing all declining pages would take about {weeks_to_review_all_declining:.1f} weeks")# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Total pages: 30000
Declining pages: 16262
If a reviewer checks 50 pages/week, reviewing all declining pages would take about 325.2 weeks


## 3. Quick look at the data (2-3 real numbers)


Looking at the starter dataset (30,000 pages across 32 clients), a few numbers support Lane 2 as worth pursuing:

16,262 pages (54.2%) are currently trending "down" — a large, non-rare pool of candidates for a review queue, unlike the sparse AI-referral direction the lane guide warns against.
1,205 pages have no position data (avg_position == 0), confirming the "0 means missing, not rank zero" gotcha — this needs handling before any modeling, not fillna(0).
Content is heavily skewed toward one type: 27,207 of 30,000 pages (about 91%) are "keyword article," with far fewer "feedly article" (2,096) and "comparison article" (697) pages. This tells me content_type will need careful handling — any pattern I find could just reflect "keyword articles behave differently," not something meaningful about performance in general.

In [24]:
print("Rows, columns:", df.shape)
print("Unique clients:", df["client_id"].nunique())

# Real number 1: how many pages are declining right now?
declining_count = (df["trend_direction"] == "down").sum()
declining_pct = declining_count / len(df) * 100
print(f"Declining pages: {declining_count} ({declining_pct:.1f}% of all pages)")

# Real number 2: how many pages have no position data (avg_position == 0)?
no_data_count = (df["avg_position"] == 0).sum()
print(f"Pages with no position data: {no_data_count}")

# Real number 3: content_type breakdown
print(df["content_type"].value_counts())

Rows, columns: (30000, 44)
Unique clients: 32
Declining pages: 16262 (54.2% of all pages)
Pages with no position data: 1205
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


## 4. Careful words: what I can and can't claim

This work will produce observed and decision-support results only: a ranked list based on patterns in historical data, meant to guide where a human looks first — not a guarantee of what will happen to any individual page. It will not claim to know Google's ranking algorithm, prove that a refresh causes recovery, or predict AI search visibility. Any "confidence" score reflects how strongly a page matches the pattern of past reviewed pages — not certainty about the future.

In [25]:
# Section 4 — document and check "unsafe" columns for future modeling
# These must NEVER be used as model features (per the flyrank-data skill's leakage warning)
unsafe_columns = ["trend_direction", "trend_pct"]

safe_columns = [col for col in df.columns if col not in unsafe_columns]

print("Unsafe columns (label-derived, excluded from any future model):", unsafe_columns)
print(f"Safe columns available for future feature-building: {len(safe_columns)} of {len(df.columns)} total")

Unsafe columns (label-derived, excluded from any future model): ['trend_direction', 'trend_pct']
Safe columns available for future feature-building: 42 of 44 total


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.